In [6]:
## Running the Simulation
def run_simulation(a, L, N, t_inital, t_final, dt, alpha, beta, vd_norm, G, np_potential, analytical_sol, forcing_fn, stablising_steps=10, mode="rest", verbose=10):
    """Runs the simulation for the normalised forced KdV equation using the Crank-Nicolson scheme. The simulation can be run in two modes: 'rest' for starting from rest frame, and 'analyitcal' starting from some exact solution for the kdv equation. The function returns the solution array, time array, and mesh space."""

    def _get_next_solution(prev_sol, curr_sol, sim_time, dt):
        """Helper function to compute the next solution using the Crank-Nicolson scheme."""
        forcing = forcing_fn(np_potential=np_potential, x=x, time=sim_time, vd=vd_norm, G=G)
        return ck_solver(last_sol=prev_sol,
                         current_sol=curr_sol,
                         alpha=alpha,
                         beta=beta,
                         k=k,
                         f=forcing,
                         dt=dt
        )

    x = np.linspace(-L, L, N, endpoint=False) / (plasma_prop["Debye Length"] * 1e-2)  # Defining mesh space & wave numbers
    dx = x[1] - x[0]
    k = 2 * np.pi * np.fft.fftfreq(N, d=dx)

    if mode == "rest":
        num_allocated_steps = int(abs(t_inital - t_final) / dt) - 1 + stablising_steps
    elif mode == "analytical":
        num_allocated_steps = int(abs(t_inital - t_final) / dt) + 2
    else:
        raise ValueError("Invalid mode. Please choose 'rest' or 'analytical'.")

    time_array = np.zeros(num_allocated_steps) # Pre-allocate time array for efficiency
    sol = np.zeros((len(time_array), N)) # Pre-allocate solution array for efficiency

    step = 1  # Start from the second time step since the first two are already set
    recorded_steps = 0  # Counter for the number of recorded steps
    sim_time = t_inital  # start
    while sim_time <= t_final:
        if mode == "rest" and sim_time < t_inital + dt:
            prev_sol, curr_sol = np.zeros(N), np.zeros(N)  # Initial condition is set to 0

            while sim_time < t_inital + dt: # stabilising the solution for the first time step to avoid numerical instability
                sim_time += dt / stablising_steps
                next_sol = _get_next_solution(prev_sol, curr_sol, sim_time, dt/stablising_steps)
                prev_sol, curr_sol= curr_sol, next_sol
            prev_sol = np.zeros(N)  # Set the previous solution back to U0 to maintian fixed dt
            print(f"Stabilisation complete. Time: {sim_time:.2f} TU")
        elif sim_time == t_inital:
            sim_time += dt
            prev_sol, curr_sol = analytical_sol(x = x), analytical_sol(x = x)  # Initial condition is set to the exact solution of the KdV equation
        else:
            sim_time += dt

        next_sol = _get_next_solution(prev_sol, curr_sol, sim_time, dt)

        if step % verbose == 0:  # only prints and records every verbose number of steps
            sol[recorded_steps] = next_sol
            time_array[recorded_steps] = sim_time

            recorded_steps += 1  # Increment the recorded steps counter
            print(f"Time: {sim_time:.2f} TU")
            print(f"Max value of solution: {np.max(sol[recorded_steps]):.4f}, Min value of solution: {np.min(sol[recorded_steps]):.4f}")

        prev_sol, curr_sol= curr_sol, next_sol
        step += 1

    print("Simulation complete.")
    return sol[:recorded_steps], time_array[:recorded_steps], x

def fkdv_exact_solution(a = 12, x = None, K = 0.6124):
    return a / (np.cosh(K * x)**2)

#analytical_sol, analytical_time_array, xi= run_simulation( a, L, N, tau_inital, tau_final, dtau, alpha, beta, vd_norm, G, np_potential,fkdv_exact_solution, forcing_term_general, mode="analytical", verbose=1000)

numerical_sol, numerical_time_array, _ = run_simulation(a, L, N, tau_inital, tau_final, dtau, alpha, beta, vd_norm, G, np_potential,fkdv_exact_solution, forcing_term_general, mode="rest", verbose=1000)

print(numerical_sol)

Stabilisation complete. Time: 0.00 TU
Time: 0.10 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 0.20 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 0.30 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 0.40 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 0.50 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 0.60 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 0.70 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 0.80 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 0.90 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 1.00 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 1.10 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 1.20 TU
Max value of solution: 0.0000, Min value of solution: 0.0000
Time: 1.30 TU
Max value of solution: 0.0000, Min value of solu

# Plasma solution simulation



## Functions

In [7]:
def initialise_ionosphere_region(df, region):
    """ takes a dataframe of ionospheric properties and pulls the associated properties for a given region. making the necessary calculations required to run simulation CURRENTLY NOT ABLE TO HANDLE GEO"""

    # Pulls property name and values into a list and numpy array respectively
    value = df_ionosphere[region].to_numpy()
    properties = df_ionosphere["Properties"].to_list()

    # Creates a dictionary of the properties and values for the given region
    plasma_properties = dict(zip(properties, value))
    return plasma_properties


def normalised_plasma_potential(a, debye_length, mass_ratio, potential_radius=None):
    """Calculates the normalised plasma potential for a given debris radius, debye length, and mass ratio."""

    def _eqn_surface_potential(x, mass_ratio, s, a):
        """Equation 32 from the NASA paper in the form 0 = f(x) where f(x) is equal to g(x) - A(1-B * e(Cx), this equation is then put through brents method to identify the roots of the function, eφ/kT"""
        return np.exp(x) - np.sqrt(mass_ratio) * (s / a) ** 2 * (
                    1 - ((s ** 2 - a ** 2) / s ** 2) * np.exp((a ** 2 / (s ** 2 - a ** 2)) * x))

    # If potential_radius is not provided, it defaults to the debye_length.
    if potential_radius is None:
        potential_radius = debye_length

    s = a + debye_length

    x_sol = brentq(_eqn_surface_potential, -10, 0, args=(mass_ratio, s, a))
    phi_np = x_sol * np.exp(-potential_radius / debye_length)  # normalised plasma potential

    return phi_np


def forcing_term_general(np_potential=None, x=None, time=None, vd=None, G=None):
    """ Calculates the forcing term for the normalised forced KdV equation
        Parameters: X IS XI AND TIME IS TAU
        Returns: Forcing term as a numpy array
    """
    return np_potential * np.exp(-1 * ((x - (vd * time)) / G) ** 2)


def ck_solver(last_sol, current_sol, alpha, beta, k, f, dt):
    """Use the Crank-Nicolson scheme to solve the normalised forced KdV equation on a non-periodic domain."""
    # pre-calculating coefficients
    _ck_beta = dt * beta * 1j * k ** 3  # removed s**3 term
    _ck_z = dt * 1j * k  # removed s term

    # computing the rfft of u & f halfs the length of array which then is multipled by 1024 leng coefficents arrays because of k
    _ck_beta_fft = _ck_beta[:len(k) // 2 + 1]  # Keep only first 513 elements
    _ck_z_fft = _ck_z[:len(k) // 2 + 1]

    # Fourier transform of the last solution, current solution, and forcing term
    fft_last = rfft(last_sol)
    fft_current_sqrd = rfft(current_sol ** 2)
    fft_f = rfft(f)

    return irfft(((1 - _ck_beta_fft) ** -1) * ((1 + _ck_beta_fft) * fft_last - alpha * _ck_z_fft * fft_current_sqrd + _ck_z_fft * fft_f))

## Imports & variables

In [8]:
## Imports & variables
import numpy as np
from numpy.fft import rfft, irfft
import pandas as pd
import matplotlib.pyplot as plt
from IPython.core.pylabtools import figsize
from scipy.optimize import brentq

df_ionosphere = pd.read_excel(r'Ionosphere prop.xltx') # Upload the Ionosphere properties from the Excel file
plasma_prop = initialise_ionosphere_region(df_ionosphere, "Low LEO") # Pulls the properties for the Low LEO region into a dictionary
display(df_ionosphere)

# Global constants
__kb__ = 1.380649e-23  #Boltzmann constant in J/K
__c__ = 299792458.0  # Speed of light in m/s
__me__ = 9.1093837139e-31  # Electron mass in kg
__mh__ = 1.67262192595e-27  # Hydrogen ion aka proton mass in kg
__mo__ = 15.994914619 * 1.66053906892e-27  # Oxygen ion mass in kg

# Normlaised units
TU = (plasma_prop["Debye Length"] * 1e-2) / (plasma_prop["Ion Acoustic Velocity"] * 1e3)  # Time unit in seconds
vi =  plasma_prop["Ion Acoustic Velocity"]  # Ion acoustic velocity in m/s

# ======================================= Inputs ====================================================

# physical parameters
a = 0.01  # debris in m
alpha = 1.0051 # Nonlinearity parameter ( 1.0051)
beta = 0.4925 # Dispersion parameter (0.4925)

# forcing parameters
vd = 1.20  # Debris velocity in km/s
np_potential = None  # Normalised plasma potential, if None, it will be automatically calculated.
G = a / (plasma_prop["Debye Length"] * 1e-2)  # Normalised forcing width

# Simulation parameters
N = 1024  # number of discretised points in the domain (power of 2 for FFT speed)
L = 300.0  # size of the domain
dtau = 1e-4  # TU
tau_inital = 0.0  # TU
tau_final =  60.0   # TU

# ======================================================================================================

vd_norm = vd / vi  # Normalised debris velocity

# Determine the ion species and set the corresponding ion mass and mass ratio
if plasma_prop["Ion Species"] == "O":
    mi = __mo__
    mass_ratio = __me__ / __mo__
elif plasma_prop["Ion Species"] == "H":
    mi = __mh__
    mass_ratio = __me__ / __mh__
else:
    raise ValueError("Ion species not recognised. Please use 'O' for Oxygen or 'H' for Hydrogen.")

# if up
if np_potential is None:
    np_potential = normalised_plasma_potential(a, plasma_prop["Debye Length"], mass_ratio)

,Properties,Units,Low LEO,High LEO I,High LEO II,GEO
0,Altitude,km,700,1200,1400,35786
1,Latitude,deg,"0,50",0-55,0-55,0-90
2,Electron Density,cm^-3,200000,100000,10000,1
3,Electron Tempreture,eV,0.25,0.35,0.35,1000
4,Ion Species,NaN,O,H,H,OH
5,Ion Mass,proton mass,16,1,1,"16,1"
6,Debye Length,cm,0.83,1.38,4.36,23504
7,Debris Velocity Circular Orbit,km/s,7.51,7.26,7.16,3.08
8,Ion Acoustic Velocity,km/s,1.22,5.75,5.75,"77.43 , 309.73"


## Running the Simulation

### Comparing the numerical and analytical solutions

In [9]:
plt.figure(figsize=(8, 6))

time = 58

for i_analytical in range(len(analytical_time_array)):
    if abs(analytical_time_array[i_analytical] - time) < 1e-2:
        print(i_analytical)
        break
for i_numerical in range(len(numerical_time_array)):
    if abs(numerical_time_array[i_numerical] - time) < 1e-2:
        print(i_numerical)
        break

print(numerical_time_array[i_numerical], analytical_time_array[i_analytical])
plt.plot(xi, numerical_sol[i_numerical] - analytical_sol[i_analytical], label="Difference between numerical and analytical solutions")
plt.xlabel("x")
plt.ylabel("Unumerical - Uanalytical")
plt.grid()
plt.title("Comparison of Numerical and Analytical Solutions at Final Time Step")

NameError: name 'analytical_time_array' is not defined

<Figure size 800x600 with 0 Axes>

## Validating the simulation

## Visualisation

### 2D Plot of U & X at some time

In [ ]:
print("Plotting Numerical Solution.")
# ---Display the final solution at the last time step---
for i in range(len(numerical_time_array)):
    if abs(numerical_time_array[i] - 60) < 1e-2:
        print(i)
        break
plt.figure(figsize=(6,5))
plt.plot(xi, numerical_sol[i], label=f"t = {numerical_time_array[i]:.2f} TU")
plt.axvline(vd_norm * numerical_time_array[i] , color='red', linestyle='--', label="Debris position")
plt.xlabel("x")
plt.ylabel("U")
plt.ylim(-1.5,3)
plt.legend()
plt.grid()
plt.show()

print("Plotting Analytical Solution.")
# ---Display the final solution at the last time step---
for i in range(len(analytical_time_array)):
    if abs(analytical_time_array[i] - 60) < 1e-2:
        print(i)
        break
plt.figure(figsize=(6,5))
plt.plot(xi, analytical_sol[i], label=f"t = {analytical_time_array[i]:.2f} TU")
plt.axvline(vd_norm * analytical_time_array[i] , color='red', linestyle='--', label="Debris position")
plt.xlabel("x")
plt.ylabel("U")
plt.ylim(-1.5,3)
plt.legend()
plt.grid()
plt.show()

### 3D Surface Plot of U, X, & T

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

Xi, Tau = np.meshgrid(xi, time_array)  # Create meshgrid for plotting
fig = plt.figure(figsize=(6,5))
ax = plt.subplot(111, projection='3d')
surf = ax.plot_surface(Xi, Tau, sol, rstride=1000, cstride=1000, cmap='viridis', linewidth=0.1,antialiased=True)

ax.set_xlabel('x axis')
ax.set_ylabel('t axis')
ax.set_zlabel('U axis')

ax.view_init(elev=23, azim=100, roll=0)

plt.show()